%md
# 02. BNPL Data Engineering

## Project
**Credit Loss Forecasting and Stress Testing for BNPL Lending**

## Objective

This notebook implements the distributed data-engineering pipeline for the synthetic Nigerian BNPL dataset.

The pipeline transforms the source data through three analytical layers:

**Source → Bronze → Silver → Gold**

### Bronze Layer

The Bronze layer provides a durable ingestion layer that preserves the source records while recording ingestion metadata and lineage information.

### Silver Layer

The Silver layer standardises data types, cleans categorical fields, validates business rules and removes only records that fail defined structural checks.

### Gold Layer

The Gold layer contains transaction-level analytical data and point-in-time-safe customer behavioural features for downstream:

- Probability of Default modelling
- Behavioural versus Full Information model comparison
- Customer segmentation
- Portfolio risk analysis
- Expected-loss analysis
- Stress testing

## Point-in-Time Requirement

Historical behavioural features must use only information available before the current transaction.

For transaction `t`, no feature may use:

- the current transaction's outcome as historical information
- a future transaction
- a future default outcome
- any observation occurring after the prediction transaction

Where the source contains only a transaction date rather than a transaction timestamp, transactions occurring on the same date are treated conservatively as simultaneous for rolling time-window calculations.

## Data Limitation

The Nigerian BNPL dataset is synthetic. Engineered features, model outputs and risk relationships therefore describe the supplied synthetic portfolio and must not be interpreted as empirical estimates of actual Nigerian BNPL behaviour.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [0]:
# ============================================================
# SOURCE INGESTION
# ============================================================

csv_path = "/Volumes/workspace/default/bnpl_raw_csv/nigerian_bnpl_full.xls"

raw_bnpl_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

print("Source rows:", raw_bnpl_df.count())
print("Source columns:", len(raw_bnpl_df.columns))

raw_bnpl_df.printSchema()

Source rows: 2000000
Source columns: 16
root
 |-- transaction_id: string (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- principal_ngn: double (nullable = true)
 |-- interest_rate_monthly: double (nullable = true)
 |-- tenor_days: integer (nullable = true)
 |-- num_installments: integer (nullable = true)
 |-- provider: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- first_time_customer: boolean (nullable = true)
 |-- first_payment_due: date (nullable = true)
 |-- default_30d: boolean (nullable = true)
 |-- default_90d: boolean (nullable = true)



In [0]:
# ============================================================
# STANDARDISE SOURCE DATE FIELDS
# ============================================================

source_bnpl_df = (
    raw_bnpl_df
    .withColumn(
        "purchase_date",
        F.to_timestamp("purchase_date")
    )
    .withColumn(
        "first_payment_due",
        F.to_timestamp("first_payment_due")
    )
)

print("Source dataset after date standardisation:")
source_bnpl_df.printSchema()

Source dataset after date standardisation:
root
 |-- transaction_id: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- principal_ngn: double (nullable = true)
 |-- interest_rate_monthly: double (nullable = true)
 |-- tenor_days: integer (nullable = true)
 |-- num_installments: integer (nullable = true)
 |-- provider: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- first_time_customer: boolean (nullable = true)
 |-- first_payment_due: timestamp (nullable = true)
 |-- default_30d: boolean (nullable = true)
 |-- default_90d: boolean (nullable = true)



In [0]:
# ============================================================
# SOURCE INTEGRITY CHECKPOINT
# ============================================================

source_rows = source_bnpl_df.count()
source_unique_transactions = (
    source_bnpl_df
    .select("transaction_id")
    .distinct()
    .count()
)

source_duplicate_transactions = (
    source_bnpl_df
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Source rows:", source_rows)
print("Unique transaction IDs:", source_unique_transactions)
print("Duplicate transaction IDs:", source_duplicate_transactions)

assert source_duplicate_transactions == 0, (
    "Duplicate transaction IDs detected in the source dataset."
)

Source rows: 2000000
Unique transaction IDs: 2000000
Duplicate transaction IDs: 0


In [0]:
# ============================================================
# CREATE SPARK-COMPATIBLE PARQUET
# ============================================================

fixed_parquet_path = (
    "/Volumes/workspace/default/bnpl_raw/"
    "bnpl_spark_compatible_parquet"
)

(
    source_bnpl_df
    .write
    .mode("overwrite")
    .parquet(fixed_parquet_path)
)

print("Spark-compatible Parquet created successfully.")
print("Path:", fixed_parquet_path)

Spark-compatible Parquet created successfully.
Path: /Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet


In [0]:
# ============================================================
# VERIFY SPARK-COMPATIBLE PARQUET
# ============================================================

fixed_df = spark.read.parquet(fixed_parquet_path)

fixed_rows = fixed_df.count()
fixed_columns = len(fixed_df.columns)

print("Parquet rows:", fixed_rows)
print("Parquet columns:", fixed_columns)

fixed_df.printSchema()

assert fixed_rows == source_rows, (
    "Row-count mismatch between source and Spark-compatible Parquet."
)

Parquet rows: 2000000
Parquet columns: 16
root
 |-- transaction_id: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- principal_ngn: double (nullable = true)
 |-- interest_rate_monthly: double (nullable = true)
 |-- tenor_days: integer (nullable = true)
 |-- num_installments: integer (nullable = true)
 |-- provider: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- first_time_customer: boolean (nullable = true)
 |-- first_payment_due: timestamp (nullable = true)
 |-- default_30d: boolean (nullable = true)
 |-- default_90d: boolean (nullable = true)



%md
# Bronze Layer

The Bronze layer is the first durable Delta representation of the source dataset.

Its purpose is to:

- preserve source records
- maintain data lineage
- record ingestion metadata
- provide a reproducible downstream input

No behavioural or machine-learning features are created in Bronze.

In [0]:
# ============================================================
# BRONZE LAYER
# ============================================================

bronze_path = "/Volumes/workspace/default/bnpl_raw/bronze_bnpl"

bronze_df = (
    fixed_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_path",
        F.lit(fixed_parquet_path)
    )
)

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(bronze_path)
)

print("Bronze Delta layer created successfully.")
print("Path:", bronze_path)

Bronze Delta layer created successfully.
Path: /Volumes/workspace/default/bnpl_raw/bronze_bnpl


In [0]:
# ============================================================
# BRONZE VALIDATION
# ============================================================

bronze_check = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

bronze_rows = bronze_check.count()

print("Bronze rows:", bronze_rows)
print("Bronze columns:", len(bronze_check.columns))

display(
    bronze_check.select(
        "transaction_id",
        "purchase_date",
        "customer_id",
        "_ingestion_timestamp",
        "_source_path"
    ).limit(10)
)

assert bronze_rows == source_rows, (
    "Bronze row count does not match the source dataset."
)

Bronze rows: 2000000
Bronze columns: 18


transaction_id,purchase_date,customer_id,_ingestion_timestamp,_source_path
BNPL-0000274330,2024-07-10T00:00:00.000Z,CUS-00036437,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0001873807,2023-08-01T00:00:00.000Z,CUS-00131005,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0001029126,2022-02-12T00:00:00.000Z,CUS-00648656,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0001512791,2023-01-26T00:00:00.000Z,CUS-00611557,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0000552055,2023-01-12T00:00:00.000Z,CUS-00499951,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0001320205,2024-10-31T00:00:00.000Z,CUS-00361857,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0001240716,2022-07-14T00:00:00.000Z,CUS-00017739,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0001905652,2024-12-18T00:00:00.000Z,CUS-00607469,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0000756202,2024-08-05T00:00:00.000Z,CUS-00655563,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet
BNPL-0000173103,2022-06-29T00:00:00.000Z,CUS-00535542,2026-09-10T19:36:15.974Z,/Volumes/workspace/default/bnpl_raw/bnpl_spark_compatible_parquet


%md
## Bronze Interpretation

The Bronze layer preserves the complete source population while adding technical lineage metadata.

The row count is reconciled against the source ingestion checkpoint, providing an explicit engineering control between ingestion and storage.

The Bronze layer contains no predictive or behavioural feature engineering. This separation ensures that analytical transformations occur downstream in Silver and Gold rather than altering the raw ingestion layer.

In [0]:
# ============================================================
# SILVER PRE-CLEANING QUALITY AUDIT
# ============================================================

silver_quality = bronze_check.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("transaction_id").alias("unique_transactions"),
    F.countDistinct("customer_id").alias("unique_customers"),

    F.sum(
        F.when(F.col("transaction_id").isNull(), 1).otherwise(0)
    ).alias("null_transaction_id"),

    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("null_customer_id"),

    F.sum(
        F.when(F.col("purchase_date").isNull(), 1).otherwise(0)
    ).alias("null_purchase_date"),

    F.sum(
        F.when(F.col("principal_ngn").isNull(), 1).otherwise(0)
    ).alias("null_principal"),

    F.sum(
        F.when(F.col("credit_score").isNull(), 1).otherwise(0)
    ).alias("null_credit_score"),

    F.sum(
        F.when(F.col("default_90d").isNull(), 1).otherwise(0)
    ).alias("null_target"),

    F.sum(
        F.when(F.col("principal_ngn") <= 0, 1).otherwise(0)
    ).alias("nonpositive_principal"),

    F.sum(
        F.when(F.col("interest_rate_monthly") < 0, 1).otherwise(0)
    ).alias("negative_interest"),

    F.sum(
        F.when(F.col("tenor_days") <= 0, 1).otherwise(0)
    ).alias("invalid_tenor"),

    F.sum(
        F.when(F.col("num_installments") <= 0, 1).otherwise(0)
    ).alias("invalid_installments"),

    F.sum(
        F.when(
            (F.col("credit_score") < 300) |
            (F.col("credit_score") > 850),
            1
        ).otherwise(0)
    ).alias("invalid_credit_score"),

    F.sum(
        F.when(
            F.col("first_payment_due") < F.col("purchase_date"),
            1
        ).otherwise(0)
    ).alias("payment_due_before_purchase"),

    F.sum(
        F.when(
            F.col("default_30d") &
            ~F.col("default_90d"),
            1
        ).otherwise(0)
    ).alias("default_30_without_90")
)

display(silver_quality)

total_rows,unique_transactions,unique_customers,null_transaction_id,null_customer_id,null_purchase_date,null_principal,null_credit_score,null_target,nonpositive_principal,negative_interest,invalid_tenor,invalid_installments,invalid_credit_score,payment_due_before_purchase,default_30_without_90
2000000,2000000,633356,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# ============================================================
# SILVER CLEANING AND STANDARDISATION
# ============================================================

silver_base = (
    bronze_check

    # Standardise categorical fields
    .withColumn(
        "merchant_category",
        F.trim(F.col("merchant_category"))
    )
    .withColumn(
        "merchant_name",
        F.trim(F.col("merchant_name"))
    )
    .withColumn(
        "customer_state",
        F.trim(F.col("customer_state"))
    )
    .withColumn(
        "provider",
        F.trim(F.col("provider"))
    )

    # Standardise numerical types
    .withColumn(
        "principal_ngn",
        F.col("principal_ngn").cast("double")
    )
    .withColumn(
        "interest_rate_monthly",
        F.col("interest_rate_monthly").cast("double")
    )
    .withColumn(
        "tenor_days",
        F.col("tenor_days").cast("int")
    )
    .withColumn(
        "num_installments",
        F.col("num_installments").cast("int")
    )
    .withColumn(
        "credit_score",
        F.col("credit_score").cast("int")
    )
)

# ============================================================
# DEFINE VALID RECORDS
# ============================================================

valid_record = (
    F.col("transaction_id").isNotNull()
    & F.col("customer_id").isNotNull()
    & F.col("purchase_date").isNotNull()
    & F.col("principal_ngn").isNotNull()
    & (F.col("principal_ngn") > 0)
    & F.col("interest_rate_monthly").isNotNull()
    & (F.col("interest_rate_monthly") >= 0)
    & F.col("tenor_days").isNotNull()
    & (F.col("tenor_days") > 0)
    & F.col("num_installments").isNotNull()
    & (F.col("num_installments") > 0)
    & F.col("credit_score").isNotNull()
    & (F.col("credit_score") >= 300)
    & (F.col("credit_score") <= 850)
    & F.col("default_90d").isNotNull()
    & (
        F.col("first_payment_due").isNull()
        | (
            F.col("first_payment_due")
            >= F.col("purchase_date")
        )
    )
    & ~(
        F.col("default_30d")
        & ~F.col("default_90d")
    )
)

silver_valid = silver_base.filter(valid_record)

# ============================================================
# DUPLICATE HANDLING
# ============================================================

# The source audit establishes transaction ID uniqueness.
# This remains as a defensive control in Silver.

silver_df = (
    silver_valid
    .dropDuplicates(["transaction_id"])
    .drop(
        "_ingestion_timestamp",
        "_source_path"
    )
)

print("Silver rows:", silver_df.count())
print("Silver columns:", len(silver_df.columns))

Silver rows: 2000000
Silver columns: 16


In [0]:
# ============================================================
# SILVER VALIDATION
# ============================================================

silver_rows = silver_df.count()
silver_unique_transactions = (
    silver_df
    .select("transaction_id")
    .distinct()
    .count()
)

silver_duplicate_ids = (
    silver_df
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Silver rows:", silver_rows)
print("Silver unique transaction IDs:", silver_unique_transactions)
print("Silver duplicate transaction IDs:", silver_duplicate_ids)

display(
    silver_df.select(
        "transaction_id",
        "customer_id",
        "purchase_date",
        "principal_ngn",
        "interest_rate_monthly",
        "tenor_days",
        "num_installments",
        "credit_score",
        "default_90d"
    ).limit(10)
)

assert silver_duplicate_ids == 0
assert silver_rows <= bronze_rows

Silver rows: 2000000
Silver unique transaction IDs: 2000000
Silver duplicate transaction IDs: 0


transaction_id,customer_id,purchase_date,principal_ngn,interest_rate_monthly,tenor_days,num_installments,credit_score,default_90d
BNPL-0000579074,CUS-00035101,2023-02-27T00:00:00.000Z,42645.14570952124,0.03246856030776745,30,1,529,false
BNPL-0000111627,CUS-00603558,2022-05-04T00:00:00.000Z,6463.785338197611,0.0,14,1,488,false
BNPL-0001310086,CUS-00494797,2024-05-20T00:00:00.000Z,27132.59798773449,0.02887669139264165,90,3,689,false
BNPL-0000777994,CUS-00394808,2022-10-11T00:00:00.000Z,41999.1853177013,0.02620805556246172,30,1,697,false
BNPL-0001537799,CUS-00270329,2023-12-30T00:00:00.000Z,15790.686828443422,0.03523678499512115,30,1,671,false
BNPL-0000237919,CUS-00268310,2022-04-27T00:00:00.000Z,8521.170182024758,0.033104395703190764,30,1,597,false
BNPL-0000511957,CUS-00011988,2023-07-19T00:00:00.000Z,62976.04582892567,0.03975275787880813,30,1,614,false
BNPL-0000034580,CUS-00602327,2024-06-29T00:00:00.000Z,49699.53670875018,0.0,14,1,686,false
BNPL-0000064138,CUS-00141684,2024-01-17T00:00:00.000Z,62995.242743252566,0.028332431562974523,30,1,579,false
BNPL-0001526238,CUS-00065241,2022-06-21T00:00:00.000Z,236674.62513127096,0.0,60,2,477,false


## Silver Interpretation

The Silver layer converts the ingested data into a clean analytical representation.

The following controls are applied:

- Data types are standardised
- Categorical text fields are trimmed
- Required identifiers and modelling fields are validated
- Transaction exposure and repayment variables are checked for valid ranges
- Credit scores are restricted to the expected 300 to 850 range
- Payment dates cannot precede purchase dates
- `default_30d = 1` cannot occur without `default_90d = 1`
- Duplicate transaction IDs are explicitly checked

Only records failing defined structural rules are eligible for removal.

For the supplied synthetic dataset, the preceding audit indicates that these validity checks should produce no material row loss. This is consistent with the clean structure identified in the initial data audit.

The Silver layer therefore provides the controlled input for point-in-time Gold feature engineering.

In [0]:
# ============================================================
# WRITE SILVER DELTA
# ============================================================

silver_path = "/Volumes/workspace/default/bnpl_raw/silver_bnpl"

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path)
)

print("Silver Delta layer created successfully.")
print("Path:", silver_path)

Silver Delta layer created successfully.
Path: /Volumes/workspace/default/bnpl_raw/silver_bnpl


In [0]:
# ============================================================
# READ SILVER DELTA
# ============================================================

silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

print("Silver rows:", silver_df.count())
print("Silver columns:", len(silver_df.columns))

silver_df.printSchema()

Silver rows: 2000000
Silver columns: 16
root
 |-- transaction_id: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- principal_ngn: double (nullable = true)
 |-- interest_rate_monthly: double (nullable = true)
 |-- tenor_days: integer (nullable = true)
 |-- num_installments: integer (nullable = true)
 |-- provider: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- first_time_customer: boolean (nullable = true)
 |-- first_payment_due: timestamp (nullable = true)
 |-- default_30d: boolean (nullable = true)
 |-- default_90d: boolean (nullable = true)



%md
# Gold Layer: Point-in-Time BNPL Behavioural Features

The Gold layer creates transaction-level analytical features for downstream modelling and risk analysis.

## Point-in-Time Design

Customer transaction history is ordered deterministically using:

`purchase_date → transaction_id`

This provides a stable ordering when multiple transactions share the same purchase date.

For cumulative historical features, only transactions preceding the current transaction are included.

For rolling time-window features, the current transaction is excluded and only transactions from the preceding 30, 60 or 90 calendar days are considered.

Because the source provides a transaction date rather than a true event timestamp, same-day observations are conservatively excluded from rolling time-window calculations.

## Historical Default Maturity

The target `default_90d` represents a 90-day future outcome.

Therefore, a previous transaction can only contribute to historical default features after its 90-day observation period has matured.

The Gold layer therefore distinguishes:

- Prior transaction history
- Prior cumulative exposure
- Prior matured transaction history
- Prior matured 90-day defaults
- Prior matured default rate
- Previous transaction date
- Days since previous transaction
- Rolling transaction frequency
- Rolling exposure

No current or future `default_90d` outcome is used as a predictive feature.

These features provide the behavioural foundation for downstream Full Information versus Behavioural model comparison.

In [0]:
# ============================================================
# CELL 19: Gold - Point-in-Time Historical Features
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------
# 1. Base Gold dataframe
# ------------------------------------------------------------

gold_base = (
    silver_df
    .withColumn(
        "_purchase_ts",
        F.col("purchase_date").cast("timestamp")
    )
)

# ------------------------------------------------------------
# 2. Deterministic transaction ordering
#
# Customer history is ordered by purchase date and transaction
# ID. The transaction ID acts as a deterministic tie-breaker
# when multiple transactions share the same date.
# ------------------------------------------------------------

transaction_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("purchase_date"),
        F.col("transaction_id")
    )
)

# ------------------------------------------------------------
# 3. Historical transactions before current transaction
#
# The current transaction is excluded.
# ------------------------------------------------------------

prior_window = (
    transaction_window
    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)

# ------------------------------------------------------------
# 4. Prior transaction count
# ------------------------------------------------------------

gold_base = (
    gold_base
    .withColumn(
        "prior_transaction_count",
        F.count("*").over(prior_window)
    )
)

# ------------------------------------------------------------
# 5. Prior cumulative exposure
#
# IMPORTANT:
# The correct source column is principal_ngn.
# ------------------------------------------------------------

gold_base = (
    gold_base
    .withColumn(
        "prior_total_exposure",
        F.coalesce(
            F.sum("principal_ngn").over(prior_window),
            F.lit(0.0)
        )
    )
)

# ------------------------------------------------------------
# 6. Previous transaction date
# ------------------------------------------------------------

gold_base = (
    gold_base
    .withColumn(
        "previous_purchase_date",
        F.lag("purchase_date").over(transaction_window)
    )
)

# ------------------------------------------------------------
# 7. Days since previous transaction
#
# First transaction for a customer has no previous transaction,
# therefore the value is NULL.
# ------------------------------------------------------------

gold_base = (
    gold_base
    .withColumn(
        "days_since_previous_transaction",
        F.when(
            F.col("previous_purchase_date").isNotNull(),
            F.datediff(
                F.col("purchase_date"),
                F.col("previous_purchase_date")
            )
        ).otherwise(None)
    )
)

# ------------------------------------------------------------
# 8. Matured 90-day historical outcome window
#
# A previous transaction's default_90d outcome can only be
# known after 90 days have elapsed.
#
# Conservative rule:
#
# previous_purchase_date + 90 days < current_purchase_date
#
# The additional 1 second ensures strict separation.
# ------------------------------------------------------------

MATURED_90D_SECONDS = (90 * 24 * 60 * 60) + 1

matured_default_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_purchase_ts").cast("long")
    )
    .rangeBetween(
        Window.unboundedPreceding,
        -MATURED_90D_SECONDS
    )
)

# ------------------------------------------------------------
# 9. Number of previous transactions whose 90-day outcomes
#    are already observable
# ------------------------------------------------------------

gold_base = (
    gold_base
    .withColumn(
        "prior_matured_transaction_count",
        F.count("*").over(matured_default_window)
    )
)

# ------------------------------------------------------------
# 10. Previous matured 90-day defaults
# ------------------------------------------------------------

gold_base = (
    gold_base
    .withColumn(
        "prior_default_count",
        F.coalesce(
            F.sum(
                F.when(
                    F.col("default_90d") == True,
                    1
                ).otherwise(0)
            ).over(matured_default_window),
            F.lit(0)
        )
    )
)

# ------------------------------------------------------------
# 11. Point-in-time-safe historical default rate
# ------------------------------------------------------------

gold_base = (
    gold_base
    .withColumn(
        "prior_default_rate",
        F.when(
            F.col("prior_matured_transaction_count") > 0,
            F.col("prior_default_count")
            / F.col("prior_matured_transaction_count")
        ).otherwise(0.0)
    )
)

# ------------------------------------------------------------
# 12. Remove technical timestamp
# ------------------------------------------------------------

gold_base = gold_base.drop("_purchase_ts")

print("Point-in-time historical features created successfully.")

Point-in-time historical features created successfully.


In [0]:
# ============================================================
# CELL 20: Gold - Rolling Behavioural Features
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------
# 1. Rolling time constants
# ------------------------------------------------------------

SECONDS_30D = 30 * 24 * 60 * 60
SECONDS_60D = 60 * 24 * 60 * 60
SECONDS_90D = 90 * 24 * 60 * 60

# ------------------------------------------------------------
# 2. Prepare timestamp for range-based windows
# ------------------------------------------------------------

rolling_base = (
    gold_base
    .withColumn(
        "_purchase_ts",
        F.col("purchase_date").cast("timestamp")
    )
    .withColumn(
        "_purchase_ts_long",
        F.col("_purchase_ts").cast("long")
    )
)

# ------------------------------------------------------------
# 3. Rolling 30-day window
#
# Current transaction is excluded using -1 second.
# Same-day observations are excluded because purchase_date
# is only available at day-level granularity.
# ------------------------------------------------------------

window_30d = (
    Window
    .partitionBy("customer_id")
    .orderBy("_purchase_ts_long")
    .rangeBetween(
        -SECONDS_30D,
        -1
    )
)

# ------------------------------------------------------------
# 4. Rolling 60-day window
# ------------------------------------------------------------

window_60d = (
    Window
    .partitionBy("customer_id")
    .orderBy("_purchase_ts_long")
    .rangeBetween(
        -SECONDS_60D,
        -1
    )
)

# ------------------------------------------------------------
# 5. Rolling 90-day window
# ------------------------------------------------------------

window_90d = (
    Window
    .partitionBy("customer_id")
    .orderBy("_purchase_ts_long")
    .rangeBetween(
        -SECONDS_90D,
        -1
    )
)

# ------------------------------------------------------------
# 6. 30-day behavioural features
# ------------------------------------------------------------

rolling_base = (
    rolling_base
    .withColumn(
        "transactions_last_30d",
        F.count("*").over(window_30d)
    )
    .withColumn(
        "exposure_last_30d",
        F.coalesce(
            F.sum("principal_ngn").over(window_30d),
            F.lit(0.0)
        )
    )
)

# ------------------------------------------------------------
# 7. 60-day behavioural features
# ------------------------------------------------------------

rolling_base = (
    rolling_base
    .withColumn(
        "transactions_last_60d",
        F.count("*").over(window_60d)
    )
    .withColumn(
        "exposure_last_60d",
        F.coalesce(
            F.sum("principal_ngn").over(window_60d),
            F.lit(0.0)
        )
    )
)

# ------------------------------------------------------------
# 8. 90-day behavioural features
# ------------------------------------------------------------

rolling_base = (
    rolling_base
    .withColumn(
        "transactions_last_90d",
        F.count("*").over(window_90d)
    )
    .withColumn(
        "exposure_last_90d",
        F.coalesce(
            F.sum("principal_ngn").over(window_90d),
            F.lit(0.0)
        )
    )
)

# ------------------------------------------------------------
# 9. Final Gold dataframe
# ------------------------------------------------------------

gold_df = (
    rolling_base
    .drop(
        "_purchase_ts",
        "_purchase_ts_long"
    )
)

print("Rolling behavioural features created successfully.")
print("Gold rows:", gold_df.count())
print("Gold columns:", len(gold_df.columns))

Rolling behavioural features created successfully.
Gold rows: 2000000
Gold columns: 29


In [0]:
# ============================================================
# CELL 21: Gold Feature Preview
# ============================================================

display(
    gold_df.select(
        # ----------------------------------------------------
        # Transaction identifiers
        # ----------------------------------------------------
        "transaction_id",
        "customer_id",
        "purchase_date",

        # ----------------------------------------------------
        # Current transaction information
        # ----------------------------------------------------
        "principal_ngn",
        "credit_score",
        "default_90d",

        # ----------------------------------------------------
        # Historical transaction behaviour
        # ----------------------------------------------------
        "prior_transaction_count",
        "prior_total_exposure",

        # ----------------------------------------------------
        # Point-in-time-safe historical default behaviour
        # ----------------------------------------------------
        "prior_matured_transaction_count",
        "prior_default_count",
        "prior_default_rate",

        # ----------------------------------------------------
        # Recency
        # ----------------------------------------------------
        "previous_purchase_date",
        "days_since_previous_transaction",

        # ----------------------------------------------------
        # Rolling activity
        # ----------------------------------------------------
        "transactions_last_30d",
        "exposure_last_30d",

        "transactions_last_60d",
        "exposure_last_60d",

        "transactions_last_90d",
        "exposure_last_90d"
    )
    .orderBy(
        "customer_id",
        "purchase_date",
        "transaction_id"
    )
    .limit(20)
)

transaction_id,customer_id,purchase_date,principal_ngn,credit_score,default_90d,prior_transaction_count,prior_total_exposure,prior_matured_transaction_count,prior_default_count,prior_default_rate,previous_purchase_date,days_since_previous_transaction,transactions_last_30d,exposure_last_30d,transactions_last_60d,exposure_last_60d,transactions_last_90d,exposure_last_90d
BNPL-0000100874,CUS-00000001,2024-03-10T00:00:00.000Z,17716.138106108338,771,false,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0
BNPL-0001759236,CUS-00000001,2024-08-07T00:00:00.000Z,38230.946250252535,595,false,1,17716.138106108338,1,0,0.0,2024-03-10T00:00:00.000Z,150,0,0.0,0,0.0,0,0.0
BNPL-0000219008,CUS-00000002,2022-10-04T00:00:00.000Z,157905.77332618216,456,true,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0
BNPL-0000734170,CUS-00000002,2023-06-13T00:00:00.000Z,21285.63255422166,617,false,1,157905.77332618216,1,1,1.0,2022-10-04T00:00:00.000Z,252,0,0.0,0,0.0,0,0.0
BNPL-0000133882,CUS-00000003,2022-03-14T00:00:00.000Z,29734.300763336625,696,false,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0
BNPL-0001198491,CUS-00000003,2023-08-04T00:00:00.000Z,56303.07772876211,489,false,1,29734.300763336625,1,0,0.0,2022-03-14T00:00:00.000Z,508,0,0.0,0,0.0,0,0.0
BNPL-0001221362,CUS-00000004,2022-03-21T00:00:00.000Z,25989.869536603506,666,false,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0
BNPL-0000549358,CUS-00000004,2022-03-24T00:00:00.000Z,48326.328826106204,524,false,1,25989.869536603506,0,0,0.0,2022-03-21T00:00:00.000Z,3,1,25989.869536603506,1,25989.869536603506,1,25989.869536603506
BNPL-0001504493,CUS-00000005,2022-05-17T00:00:00.000Z,48428.85027296947,692,false,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0
BNPL-0001606279,CUS-00000005,2022-09-28T00:00:00.000Z,42712.970449907014,549,false,1,48428.85027296947,1,0,0.0,2022-05-17T00:00:00.000Z,134,0,0.0,0,0.0,0,0.0


In [0]:
# ============================================================
# CELL 22: Gold Feature Quality Checks
# ============================================================

from pyspark.sql import functions as F

print("============================================================")
print("GOLD FEATURE QUALITY CHECKS")
print("============================================================")

# ------------------------------------------------------------
# 1. Basic Gold layer counts
# ------------------------------------------------------------

gold_rows = gold_df.count()
gold_columns = len(gold_df.columns)

unique_transactions = (
    gold_df
    .select("transaction_id")
    .distinct()
    .count()
)

unique_customers = (
    gold_df
    .select("customer_id")
    .distinct()
    .count()
)

print("Gold rows:", gold_rows)
print("Gold columns:", gold_columns)
print("Unique transactions:", unique_transactions)
print("Unique customers:", unique_customers)

# ------------------------------------------------------------
# 2. History coverage
# ------------------------------------------------------------

history_summary = (
    gold_df
    .select(
        F.sum(
            F.when(
                F.col("prior_transaction_count") > 0,
                1
            ).otherwise(0)
        ).alias("transactions_with_prior_history"),

        F.sum(
            F.when(
                F.col("prior_transaction_count") == 0,
                1
            ).otherwise(0)
        ).alias("first_transactions"),

        F.sum(
            F.when(
                F.col("prior_matured_transaction_count") > 0,
                1
            ).otherwise(0)
        ).alias("transactions_with_matured_history")
    )
)

display(history_summary)

# ------------------------------------------------------------
# 3. Rolling activity coverage
# ------------------------------------------------------------

rolling_summary = (
    gold_df
    .select(
        F.sum(
            F.when(
                F.col("transactions_last_30d") > 0,
                1
            ).otherwise(0)
        ).alias("transactions_with_30d_activity"),

        F.sum(
            F.when(
                F.col("transactions_last_60d") > 0,
                1
            ).otherwise(0)
        ).alias("transactions_with_60d_activity"),

        F.sum(
            F.when(
                F.col("transactions_last_90d") > 0,
                1
            ).otherwise(0)
        ).alias("transactions_with_90d_activity")
    )
)

display(rolling_summary)

# ------------------------------------------------------------
# 4. Historical feature distributions
# ------------------------------------------------------------

display(
    gold_df.select(
        F.mean("prior_transaction_count")
            .alias("mean_prior_transaction_count"),

        F.mean("prior_total_exposure")
            .alias("mean_prior_total_exposure"),

        F.mean("prior_matured_transaction_count")
            .alias("mean_prior_matured_transaction_count"),

        F.mean("prior_default_count")
            .alias("mean_prior_default_count"),

        F.mean("prior_default_rate")
            .alias("mean_prior_default_rate"),

        F.mean("days_since_previous_transaction")
            .alias("mean_days_since_previous_transaction")
    )
)

# ------------------------------------------------------------
# 5. Rolling feature distributions
# ------------------------------------------------------------

display(
    gold_df.select(
        F.mean("transactions_last_30d")
            .alias("mean_transactions_last_30d"),

        F.mean("exposure_last_30d")
            .alias("mean_exposure_last_30d"),

        F.mean("transactions_last_60d")
            .alias("mean_transactions_last_60d"),

        F.mean("exposure_last_60d")
            .alias("mean_exposure_last_60d"),

        F.mean("transactions_last_90d")
            .alias("mean_transactions_last_90d"),

        F.mean("exposure_last_90d")
            .alias("mean_exposure_last_90d")
    )
)

GOLD FEATURE QUALITY CHECKS
Gold rows: 2000000
Gold columns: 29
Unique transactions: 2000000
Unique customers: 633356


transactions_with_prior_history,first_transactions,transactions_with_matured_history
1366644,633356,1210351


transactions_with_30d_activity,transactions_with_60d_activity,transactions_with_90d_activity
155586,295254,420038


mean_prior_transaction_count,mean_prior_total_exposure,mean_prior_matured_transaction_count,mean_prior_default_count,mean_prior_default_rate,mean_days_since_previous_transaction
1.5009635,75069.1184044402,1.262821,0.101063,0.04841930566378069,222.23917567413312


mean_transactions_last_30d,mean_exposure_last_30d,mean_transactions_last_60d,mean_exposure_last_60d,mean_transactions_last_90d,mean_exposure_last_90d
0.081108,4051.5107044258666,0.160077,8003.095739835053,0.2368085,11837.292493765588


In [0]:
# ============================================================
# CELL 23: Point-in-Time Leakage Validation
# ============================================================

from pyspark.sql import functions as F

print("============================================================")
print("POINT-IN-TIME LEAKAGE VALIDATION")
print("============================================================")

# ------------------------------------------------------------
# 1. No negative historical counts
# ------------------------------------------------------------

negative_prior_transactions = (
    gold_df
    .filter(F.col("prior_transaction_count") < 0)
    .count()
)

negative_matured_transactions = (
    gold_df
    .filter(F.col("prior_matured_transaction_count") < 0)
    .count()
)

negative_default_counts = (
    gold_df
    .filter(F.col("prior_default_count") < 0)
    .count()
)

negative_exposure = (
    gold_df
    .filter(F.col("prior_total_exposure") < 0)
    .count()
)

# ------------------------------------------------------------
# 2. Default count cannot exceed matured transaction count
# ------------------------------------------------------------

invalid_default_counts = (
    gold_df
    .filter(
        F.col("prior_default_count")
        > F.col("prior_matured_transaction_count")
    )
    .count()
)

# ------------------------------------------------------------
# 3. Historical default rate must be between 0 and 1
# ------------------------------------------------------------

invalid_default_rates = (
    gold_df
    .filter(
        (F.col("prior_default_rate") < 0)
        | (F.col("prior_default_rate") > 1)
    )
    .count()
)

# ------------------------------------------------------------
# 4. Previous transaction date cannot be after current date
# ------------------------------------------------------------

invalid_previous_dates = (
    gold_df
    .filter(
        F.col("previous_purchase_date")
        > F.col("purchase_date")
    )
    .count()
)

# ------------------------------------------------------------
# 5. First transactions must have no previous transaction
# ------------------------------------------------------------

invalid_first_transactions = (
    gold_df
    .filter(
        (F.col("prior_transaction_count") == 0)
        & (
            F.col("previous_purchase_date").isNotNull()
            | F.col("days_since_previous_transaction").isNotNull()
        )
    )
    .count()
)

# ------------------------------------------------------------
# 6. Rolling features cannot be negative
# ------------------------------------------------------------

negative_rolling_features = (
    gold_df
    .filter(
        (F.col("transactions_last_30d") < 0)
        | (F.col("transactions_last_60d") < 0)
        | (F.col("transactions_last_90d") < 0)
        | (F.col("exposure_last_30d") < 0)
        | (F.col("exposure_last_60d") < 0)
        | (F.col("exposure_last_90d") < 0)
    )
    .count()
)

# ------------------------------------------------------------
# 7. Display validation results
# ------------------------------------------------------------

validation_results = [
    ("negative_prior_transactions", negative_prior_transactions),
    ("negative_matured_transactions", negative_matured_transactions),
    ("negative_default_counts", negative_default_counts),
    ("negative_exposure", negative_exposure),
    ("invalid_default_counts", invalid_default_counts),
    ("invalid_default_rates", invalid_default_rates),
    ("invalid_previous_dates", invalid_previous_dates),
    ("invalid_first_transactions", invalid_first_transactions),
    ("negative_rolling_features", negative_rolling_features)
]

validation_df = spark.createDataFrame(
    validation_results,
    ["check", "violations"]
)

display(validation_df)

# ------------------------------------------------------------
# 8. Hard quality gate
# ------------------------------------------------------------

total_violations = sum(
    value for _, value in validation_results
)

assert total_violations == 0, (
    f"Point-in-time validation failed with "
    f"{total_violations} violations."
)

print("PASS: All point-in-time leakage checks returned zero violations.")

POINT-IN-TIME LEAKAGE VALIDATION


check,violations
negative_prior_transactions,0
negative_matured_transactions,0
negative_default_counts,0
negative_exposure,0
invalid_default_counts,0
invalid_default_rates,0
invalid_previous_dates,0
invalid_first_transactions,0
negative_rolling_features,0


PASS: All point-in-time leakage checks returned zero violations.


In [0]:
# ============================================================
# CELL 24: 90-Day Default Maturity Validation
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("============================================================")
print("90-DAY DEFAULT MATURITY VALIDATION")
print("============================================================")

# ------------------------------------------------------------
# Same maturity rule used in Gold construction
# ------------------------------------------------------------

MATURED_90D_SECONDS = (90 * 24 * 60 * 60) + 1

maturity_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("purchase_date").cast("timestamp").cast("long")
    )
    .rangeBetween(
        Window.unboundedPreceding,
        -MATURED_90D_SECONDS
    )
)

# ------------------------------------------------------------
# Independently reconstruct matured history
# ------------------------------------------------------------

maturity_check = (
    gold_df
    .withColumn(
        "_check_matured_count",
        F.count("*").over(maturity_window)
    )
    .withColumn(
        "_check_default_count",
        F.coalesce(
            F.sum(
                F.when(
                    F.col("default_90d") == True,
                    1
                ).otherwise(0)
            ).over(maturity_window),
            F.lit(0)
        )
    )
)

# ------------------------------------------------------------
# Compare reconstructed values with Gold features
# ------------------------------------------------------------

maturity_differences = (
    maturity_check
    .select(
        F.max(
            F.abs(
                F.col("prior_matured_transaction_count")
                - F.col("_check_matured_count")
            )
        ).alias("max_matured_count_difference"),

        F.max(
            F.abs(
                F.col("prior_default_count")
                - F.col("_check_default_count")
            )
        ).alias("max_default_count_difference")
    )
)

display(maturity_differences)

# ------------------------------------------------------------
# Hard quality gate
# ------------------------------------------------------------

maturity_result = maturity_differences.collect()[0]

max_matured_difference = maturity_result[
    "max_matured_count_difference"
]

max_default_difference = maturity_result[
    "max_default_count_difference"
]

assert max_matured_difference == 0, (
    "Matured transaction count validation failed."
)

assert max_default_difference == 0, (
    "Historical default count validation failed."
)

print("PASS: 90-day historical default maturity logic validated.")

90-DAY DEFAULT MATURITY VALIDATION


max_matured_count_difference,max_default_count_difference
0,0


PASS: 90-day historical default maturity logic validated.


In [0]:
# ============================================================
# CELL 25: Final Gold Preview
# ============================================================

display(
    gold_df.select(
        "transaction_id",
        "customer_id",
        "purchase_date",

        "principal_ngn",
        "credit_score",

        "prior_transaction_count",
        "prior_total_exposure",

        "prior_matured_transaction_count",
        "prior_default_count",
        "prior_default_rate",

        "previous_purchase_date",
        "days_since_previous_transaction",

        "transactions_last_30d",
        "exposure_last_30d",

        "transactions_last_60d",
        "exposure_last_60d",

        "transactions_last_90d",
        "exposure_last_90d",

        "default_90d"
    )
    .orderBy(
        "customer_id",
        "purchase_date",
        "transaction_id"
    )
    .limit(20)
)

transaction_id,customer_id,purchase_date,principal_ngn,credit_score,prior_transaction_count,prior_total_exposure,prior_matured_transaction_count,prior_default_count,prior_default_rate,previous_purchase_date,days_since_previous_transaction,transactions_last_30d,exposure_last_30d,transactions_last_60d,exposure_last_60d,transactions_last_90d,exposure_last_90d,default_90d
BNPL-0000100874,CUS-00000001,2024-03-10T00:00:00.000Z,17716.138106108338,771,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0,false
BNPL-0001759236,CUS-00000001,2024-08-07T00:00:00.000Z,38230.946250252535,595,1,17716.138106108338,1,0,0.0,2024-03-10T00:00:00.000Z,150,0,0.0,0,0.0,0,0.0,false
BNPL-0000219008,CUS-00000002,2022-10-04T00:00:00.000Z,157905.77332618216,456,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0,true
BNPL-0000734170,CUS-00000002,2023-06-13T00:00:00.000Z,21285.63255422166,617,1,157905.77332618216,1,1,1.0,2022-10-04T00:00:00.000Z,252,0,0.0,0,0.0,0,0.0,false
BNPL-0000133882,CUS-00000003,2022-03-14T00:00:00.000Z,29734.300763336625,696,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0,false
BNPL-0001198491,CUS-00000003,2023-08-04T00:00:00.000Z,56303.07772876211,489,1,29734.300763336625,1,0,0.0,2022-03-14T00:00:00.000Z,508,0,0.0,0,0.0,0,0.0,false
BNPL-0001221362,CUS-00000004,2022-03-21T00:00:00.000Z,25989.869536603506,666,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0,false
BNPL-0000549358,CUS-00000004,2022-03-24T00:00:00.000Z,48326.328826106204,524,1,25989.869536603506,0,0,0.0,2022-03-21T00:00:00.000Z,3,1,25989.869536603506,1,25989.869536603506,1,25989.869536603506,false
BNPL-0001504493,CUS-00000005,2022-05-17T00:00:00.000Z,48428.85027296947,692,0,0.0,0,0,0.0,null,null,0,0.0,0,0.0,0,0.0,false
BNPL-0001606279,CUS-00000005,2022-09-28T00:00:00.000Z,42712.970449907014,549,1,48428.85027296947,1,0,0.0,2022-05-17T00:00:00.000Z,134,0,0.0,0,0.0,0,0.0,false


In [0]:
# ============================================================
# CELL 26: Write Final Gold Delta
# ============================================================

(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path)
)

print("Gold Delta table written successfully.")
print("Gold path:", gold_path)

Gold Delta table written successfully.
Gold path: /Volumes/workspace/default/bnpl_raw/gold_bnpl


In [0]:
# ============================================================
# CELL 27: Reload Gold Delta
# ============================================================

gold_reload_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

print("Reloaded Gold rows:", gold_reload_df.count())
print("Reloaded Gold columns:", len(gold_reload_df.columns))

assert gold_reload_df.count() == gold_rows, (
    "Gold row count changed after persistence."
)

assert len(gold_reload_df.columns) == gold_columns, (
    "Gold column count changed after persistence."
)

print("PASS: Gold Delta persistence validated.")

Reloaded Gold rows: 2000000
Reloaded Gold columns: 29
PASS: Gold Delta persistence validated.


In [0]:
# ============================================================
# CELL 28: Final Gold Feature Gate
# ============================================================

required_gold_features = [
    "prior_transaction_count",
    "prior_total_exposure",
    "prior_matured_transaction_count",
    "prior_default_count",
    "prior_default_rate",
    "previous_purchase_date",
    "days_since_previous_transaction",
    "transactions_last_30d",
    "exposure_last_30d",
    "transactions_last_60d",
    "exposure_last_60d",
    "transactions_last_90d",
    "exposure_last_90d"
]

missing_features = [
    feature
    for feature in required_gold_features
    if feature not in gold_reload_df.columns
]

assert len(missing_features) == 0, (
    f"Missing required Gold features: {missing_features}"
)

# Features that must NOT exist because they would use immature
# 90-day future outcomes from recent transactions.
forbidden_leakage_features = [
    "defaults_last_30d",
    "defaults_last_60d",
    "defaults_last_90d"
]

present_forbidden_features = [
    feature
    for feature in forbidden_leakage_features
    if feature in gold_reload_df.columns
]

assert len(present_forbidden_features) == 0, (
    "Leakage-prone features found in Gold: "
    f"{present_forbidden_features}"
)

print("============================================================")
print("FINAL GOLD FEATURE GATE: PASS")
print("============================================================")
print("Required features:", len(required_gold_features))
print("Missing features:", len(missing_features))
print("Forbidden leakage features:", len(present_forbidden_features))

FINAL GOLD FEATURE GATE: PASS
Required features: 13
Missing features: 0
Forbidden leakage features: 0


In [0]:
# ============================================================
# CELL 29: Final Gold Layer Summary
# ============================================================

print("============================================================")
print("GOLD LAYER FINAL SUMMARY")
print("============================================================")

print("Gold rows:", gold_reload_df.count())
print("Gold columns:", len(gold_reload_df.columns))
print(
    "Unique transactions:",
    gold_reload_df.select("transaction_id").distinct().count()
)
print(
    "Unique customers:",
    gold_reload_df.select("customer_id").distinct().count()
)

print()
print("Historical features:")
print("  - Prior transaction count")
print("  - Prior cumulative exposure")
print("  - Prior matured transaction count")
print("  - Prior matured default count")
print("  - Prior matured default rate")
print("  - Previous transaction date")
print("  - Days since previous transaction")

print()
print("Rolling behavioural features:")
print("  - Transactions last 30 days")
print("  - Exposure last 30 days")
print("  - Transactions last 60 days")
print("  - Exposure last 60 days")
print("  - Transactions last 90 days")
print("  - Exposure last 90 days")

print()
print("Leakage controls:")
print("  - Current transaction excluded from historical windows")
print("  - 90-day outcomes included only after maturity")
print("  - Rolling default features excluded")
print("  - Point-in-time validation completed")

print()
print("GOLD LAYER COMPLETE.")

GOLD LAYER FINAL SUMMARY
Gold rows: 2000000
Gold columns: 29
Unique transactions: 2000000
Unique customers: 633356

Historical features:
  - Prior transaction count
  - Prior cumulative exposure
  - Prior matured transaction count
  - Prior matured default count
  - Prior matured default rate
  - Previous transaction date
  - Days since previous transaction

Rolling behavioural features:
  - Transactions last 30 days
  - Exposure last 30 days
  - Transactions last 60 days
  - Exposure last 60 days
  - Transactions last 90 days
  - Exposure last 90 days

Leakage controls:
  - Current transaction excluded from historical windows
  - 90-day outcomes included only after maturity
  - Rolling default features excluded
  - Point-in-time validation completed

GOLD LAYER COMPLETE.
